In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/satoru3825/tweet-sentiment-data/preprocess.py
/kaggle/input/datasets/satoru3825/tweet-sentiment-data/test.csv
/kaggle/input/datasets/satoru3825/tweet-sentiment-data/train.csv
/kaggle/input/datasets/satoru3825/tweet-sentiment-data/validation.csv


In [2]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files)

/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/satoru3825 []
/kaggle/input/datasets/satoru3825/tweet-sentiment-data ['preprocess.py', 'test.csv', 'train.csv', 'validation.csv']


In [3]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [4]:
import sys
import transformers

DATA = "/kaggle/input/datasets/satoru3825/tweet-sentiment-data"

sys.path.insert(0, DATA)
from preprocess import clean_tweet, preprocess

train_df = preprocess(pd.read_csv(f"{DATA}/train.csv"))
validation_df = preprocess(pd.read_csv(f"{DATA}/validation.csv"))
test_df = preprocess(pd.read_csv(f"{DATA}/test.csv"))

print(train_df.shape, validation_df.shape, test_df.shape)
print(train_df["label"].value_counts())
print("transformers:", transformers.__version__, "| torch:", torch.__version__)

train_df[["tweet", "clean_tweet", "label"]].head()

(200000, 4) (20000, 4) (20000, 4)
label
0    100000
1    100000
Name: count, dtype: int64
transformers: 5.16.1 | torch: 2.11.0+cu128


,tweet,clean_tweet,label
0,Never ever!! Eat an ice cream sandwich out of ...,never ever!! eat an ice cream sandwich out of ...,0
1,SA .. yet again,sa .. yet again,0
2,@aussiemcflyfan aww really? maybe in a few yea...,USER aww really? maybe in a few years he will ...,1
3,Im so tired!!! Baby sitting.,im so tired!!! baby sitting.,0
4,"On my way to see about getting a bus, aren't c...","on my way to see about getting a bus, aren't c...",0


In [5]:
from datasets import Dataset
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def to_dataset(df):
    encodings = tokenizer(df["clean_tweet"].tolist(), truncation=True, max_length=64)
    return Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": df["label"].tolist(),
    })

train_ds = to_dataset(train_df)
validation_ds = to_dataset(validation_df)
test_ds = to_dataset(test_df)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds[:3])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': [[101, 2196, 2412, 999, 999, 4521, 2019, 3256, 6949, 11642, 2041, 1997, 1037, 2310, 15683, 3698, 1012, 102], [101, 7842, 1012, 1012, 2664, 2153, 102], [101, 5310, 22091, 2860, 2428, 1029, 2672, 1999, 1037, 2261, 2086, 2002, 2097, 2689, 1029, 2059, 2017, 2064, 2022, 2007, 2032, 999, 102]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [0, 0, 1]}


In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

print(model.num_parameters())

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


66955010


In [7]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro"),
    }

In [8]:
import math
from transformers import TrainingArguments

num_epochs_1 = 1
train_size_1 = 50000          # the test run; 200000 for the full run
batch_size_1 = 64

total_steps_1 = math.ceil(train_size_1 / batch_size_1) * num_epochs_1
warmup_steps_1 = int(0.1 * total_steps_1)
print("total steps:", total_steps_1, "| warmup steps:", warmup_steps_1)

training_args = TrainingArguments(
    output_dir="/kaggle/working/distilbert-sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size_1,
    per_device_eval_batch_size=128,
    num_train_epochs=num_epochs_1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    warmup_steps=warmup_steps_1,
    weight_decay=0.01,
    fp16=True,
    seed=42,
    report_to="none",
    logging_steps=100,
)

total steps: 782 | warmup steps: 78


In [9]:
import time
from transformers import Trainer

small_train_ds = train_ds.shuffle(seed=42).select(range(train_size_1))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_ds,
    eval_dataset=validation_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

start = time.perf_counter()
trainer.train()
elapsed = time.perf_counter() - start

print(f"training time: {elapsed / 60:.1f} minutes")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.833618,0.787549,0.824100,0.824100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

training time: 2.3 minutes


In [10]:
num_gpus = max(1, torch.cuda.device_count())
num_epochs_2 = 2
train_size_2 = 200000
batch_size_2 = 64

steps_per_epoch = math.ceil(train_size_2 / (batch_size_2 * num_gpus))
total_steps_2 = steps_per_epoch * num_epochs_2
warmup_steps_2 = int(0.1 * total_steps_2)
print("GPUs:", num_gpus, "| total steps:", total_steps_2, "| warmup steps:", warmup_steps_2)

training_args = TrainingArguments(
    output_dir="/kaggle/working/distilbert-sentiment-full",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size_2,
    per_device_eval_batch_size=128,
    num_train_epochs=num_epochs_2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    warmup_steps=warmup_steps_2,
    weight_decay=0.01,
    fp16=True,
    seed=42,
    report_to="none",
    logging_steps=100,
)



GPUs: 2 | total steps: 3126 | warmup steps: 312


In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
) 

full_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,  
    eval_dataset=validation_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

start = time.perf_counter()
full_trainer.train()
training_minutes = (time.perf_counter() - start) / 60
print(f"training time: {training_minutes:.1f} minutes")




Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super(

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.721182,0.724977,0.840850,0.840399
2,0.641682,0.688666,0.851000,0.851000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

training time: 16.5 minutes


In [12]:
history = pd.DataFrame(full_trainer.state.log_history)
epoch_results = history.dropna(subset=["eval_accuracy"])[
    ["epoch", "eval_loss", "eval_accuracy", "eval_f1"]
]
print(epoch_results.to_string(index=False))

best = full_trainer.evaluate()

 epoch  eval_loss  eval_accuracy  eval_f1
   1.0   0.724977        0.84085 0.840399
   2.0   0.688666        0.85100 0.851000


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.641682,0.688666,2,0.851000,0.851000


In [13]:
import json

FINAL_DIR = "/kaggle/working/final_model"

full_trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

history.to_csv("/kaggle/working/training_log.csv", index=False)
epoch_results.to_csv("/kaggle/working/validation_per_epoch.csv", index=False)

info = {
    "model": model_name,
    "max_length": 64,
    "train_tweets": len(train_ds),
    "epochs": num_epochs_2,
    "batch_size_per_gpu": batch_size_2,
    "num_gpus": num_gpus,
    "gpu": torch.cuda.get_device_name(0),
    "learning_rate": 2e-5,
    "warmup_steps": warmup_steps_2,
    "training_minutes": round(training_minutes, 1),
    "best_validation_accuracy": round(best["eval_accuracy"], 4),
    "best_validation_f1": round(best["eval_f1"], 4),
}
with open("/kaggle/working/training_info.json", "w") as f:
    json.dump(info, f, indent=2)

print(json.dumps(info, indent=2))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "model": "distilbert-base-uncased",
  "max_length": 64,
  "train_tweets": 200000,
  "epochs": 2,
  "batch_size_per_gpu": 64,
  "num_gpus": 2,
  "gpu": "Tesla T4",
  "learning_rate": 2e-05,
  "warmup_steps": 312,
  "training_minutes": 16.5,
  "best_validation_accuracy": 0.851,
  "best_validation_f1": 0.851
}
